New Notebook Created by Jupyter MCP Server

In [1]:
from pathlib import Path
import os, sys, inspect
ROOT = Path('/home/nahisaho/GitHub/jupytermind')
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
os.environ['AI_DATA_SCIENTIST_PROJECTS_ROOT'] = str(ROOT / 'benchmark/projects')
from ai_data_scientist import project_manager as pm, lifecycle, language_router
handle = pm.resolve_project('kaggle-v020-kaggle-exp-09-diabetes')
pm.ensure_notebook(handle)
run_id = 'v020-exp-09'
lifecycle.register_run(run_id, handle.notebook_path)
print('言語:', language_router.detect_language('糖尿病アウトカムとの関連を分析してください'))
print('Notebook:', handle.notebook_path)
print('状態:', lifecycle.get_run_status(run_id))
from ai_data_scientist import cleaning, eda
print('clean_dataset:', inspect.signature(cleaning.clean_dataset))
print('explore:', inspect.signature(eda.explore))
print('kernel cwd:', Path.cwd())

言語: ja
Notebook: /home/nahisaho/GitHub/jupytermind/benchmark/projects/kaggle-v020-kaggle-exp-09-diabetes/notebooks/kaggle-v020-kaggle-exp-09-diabetes.ipynb
状態: RunStatus(run_id='v020-exp-09', state='running', active_cell_executions=0, pending_notebook_writes=0, locks_held=0, notebook_path='/home/nahisaho/GitHub/jupytermind/benchmark/projects/kaggle-v020-kaggle-exp-09-diabetes/notebooks/kaggle-v020-kaggle-exp-09-diabetes.ipynb', reason=None)
clean_dataset: (df: 'pd.DataFrame', operation: 'str', columns: 'list | None' = None, fill_value=None) -> 'CleaningReport'
explore: (df: 'pd.DataFrame', top_n: 'int' = 10) -> 'EDAReport'
kernel cwd: /home/nahisaho/GitHub/jupytermind/benchmark


In [2]:
import nbformat
from textwrap import dedent
setup_source = r'''# 環境・ライフサイクル（全セルをJupyter MCP経由で実行）
from pathlib import Path
import os, sys, json, hashlib, io, contextlib, re
from datetime import datetime, timezone
from dataclasses import asdict
import nbformat
ROOT = Path('/home/nahisaho/GitHub/jupytermind')
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
os.environ['AI_DATA_SCIENTIST_PROJECTS_ROOT'] = str(ROOT / 'benchmark/projects')
from ai_data_scientist import project_manager as pm, lifecycle, language_router
from ai_data_scientist import ingestion, cleaning, eda, stats_analysis, visualization
from ai_data_scientist import data_definition, analysis_assumptions, data_quality, sensitivity
from ai_data_scientist import insight_engine, notebook_audit
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
from IPython.display import display, Markdown
handle = pm.resolve_project('kaggle-v020-kaggle-exp-09-diabetes')
pm.ensure_notebook(handle)
data_dir = pm.ensure_data_dir(handle)
run_id = 'v020-exp-09'
lifecycle.register_run(run_id, handle.notebook_path)
phase_log = []
@contextlib.contextmanager
def phase(name):
    if lifecycle.is_cancel_requested(run_id):
        raise RuntimeError('キャンセル要求を検出しました')
    lifecycle.mark_execution_start(run_id)
    phase_log.append({'phase': name, 'event': 'start', 'utc': datetime.now(timezone.utc).isoformat()})
    try:
        yield
    except Exception as error:
        lifecycle.mark_failed(run_id)
        phase_log.append({'phase': name, 'event': 'failed', 'error_type': type(error).__name__})
        raise
    finally:
        lifecycle.mark_execution_end(run_id)
        phase_log.append({'phase': name, 'event': 'end', 'utc': datetime.now(timezone.utc).isoformat()})
def write_notebook(mutation):
    lifecycle.mark_write_start(run_id)
    try:
        pm.enqueue_write(handle, mutation)
    finally:
        lifecycle.mark_write_end(run_id)
print('言語:', language_router.detect_language('糖尿病アウトカムとの関連を調べてください'))
print('run_id:', run_id)
print('実行経路: Jupyter MCP。認証情報は表示・保存しない。')
'''
acquisition_source = r'''# Kaggle検索・ファイル列挙・取得（認証情報の出力禁止）
with phase('kaggle-acquisition'):
    requested_slug = 'uciml/pima-indians-diabetes-database'
    filename = 'diabetes.csv'
    stage = 'import'
    failure = None
    search_refs, file_names = [], []
    downloaded_path = data_dir / filename
    try:
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            from kaggle.api.kaggle_api_extended import KaggleApi
            api = KaggleApi()
            stage = 'authenticate'
            api.authenticate()
            stage = 'dataset_list'
            matches = api.dataset_list(search='pima indians diabetes')
            search_refs = [str(item.ref) for item in matches]
            if requested_slug not in search_refs:
                matches = api.dataset_list(search='uciml pima indians diabetes')
                search_refs = sorted(set(search_refs + [str(item.ref) for item in matches]))
            if requested_slug not in search_refs:
                raise LookupError('指定した公開データセットが検索結果に存在しない')
            stage = 'dataset_list_files'
            listing = api.dataset_list_files(requested_slug)
            file_names = [item.name for item in listing.files]
            if filename not in file_names:
                raise FileNotFoundError('指定CSVがデータセットに存在しない')
            stage = 'dataset_download_file'
            api.dataset_download_file(requested_slug, filename, path=str(data_dir), force=True, quiet=True)
            if not downloaded_path.exists():
                zipped_path = data_dir / (filename + '.zip')
                if not zipped_path.exists():
                    raise FileNotFoundError('ダウンロード後にCSVもZIPも見つからない')
                import zipfile
                with zipfile.ZipFile(zipped_path) as archive:
                    downloaded_path.write_bytes(archive.read(filename))
        source_kind = 'kaggle'
        source_url = 'https://www.kaggle.com/datasets/' + requested_slug
    except Exception as error:
        # SDK例外の本文・HTTP headers・設定・トークンは出力しない。
        status = getattr(error, 'status', None)
        failure = {'stage': stage, 'error_type': type(error).__name__,
                   'http_status': status if isinstance(status, int) else None,
                   'reason': 'Kaggle APIの該当フェーズが失敗。秘密情報保護のため例外本文は記録しない。'}
        import urllib.request
        legacy_note = Path('/home/nahisaho/kaggle/experiments/white-paper.md')
        legacy_checked = legacy_note.exists() and 'https://raw.githubusercontent.com/plotly/datasets/master/diabetes.csv' in legacy_note.read_text(encoding='utf-8')
        source_url = 'https://raw.githubusercontent.com/plotly/datasets/master/diabetes.csv'
        with urllib.request.urlopen(source_url, timeout=30) as response:
            downloaded_path.write_bytes(response.read())
        source_kind = 'fallback-public-csv'
    retrieved_utc = datetime.now(timezone.utc).isoformat()
    sha256 = hashlib.sha256(downloaded_path.read_bytes()).hexdigest()
    provenance = {'requested_owner_dataset_slug': requested_slug,
        'owner_dataset_slug': requested_slug if source_kind == 'kaggle' else None,
        'source_kind': source_kind, 'source_url': source_url,
        'retrieved_filename': filename, 'retrieved_at_utc': retrieved_utc,
        'sha256': sha256, 'search_query': 'pima indians diabetes',
        'search_results': search_refs, 'kaggle_file_names': file_names, 'kaggle_failure': failure,
        'fallback_reason': None if failure is None else 'Kaggle API検索・取得失敗時に限り指定の旧実験公開CSVを使用',
        'legacy_note_url_confirmed': None if failure is None else legacy_checked,
        'identity_warning': None if failure is None else '公開CSVとKaggle掲載版の完全な同一性は保証されない'}
    provenance_path = data_dir / 'provenance.json'
    if provenance_path.exists():
        prior_provenance = json.loads(provenance_path.read_text(encoding='utf-8'))
        print('前回取得SHAとの一致:', prior_provenance['sha256'] == sha256)
    lifecycle.mark_write_start(run_id)
    try:
        provenance_path.write_text(json.dumps(provenance, ensure_ascii=False, indent=2), encoding='utf-8')
    finally:
        lifecycle.mark_write_end(run_id)
    display(Markdown('### 取得記録\n```json\n' + json.dumps(provenance, ensure_ascii=False, indent=2) + '\n```'))
    loaded = ingestion.ingest(ingestion.SourceSpec('csv', str(downloaded_path)), row_limit=10000)
    raw = loaded.dataframe
    assert not loaded.truncated
    assert list(raw.columns) == ['Pregnancies','Glucose','BloodPressure','SkinThickness','Insulin','BMI','DiabetesPedigreeFunction','Age','Outcome']
    display(raw.head())
    print('データ形状:', raw.shape)
'''
intro = '''# Pima Indians Diabetes：糖尿病アウトカムとの関連

## 計画
run_id: `v020-exp-09`。Kaggle公開データをSDKで検索・取得し、SHA-256と取得日時で出典を固定する。型・重複・ゼロ・意味的な範囲を検査し、生理的に不自然なゼロを欠損として扱う。Outcome別の欠損率、観測値の分布、標準化群間差と信頼区間を比較し、点双列相関・調整ロジスティック回帰を行う。欠損処理と極端値処理の感度分析を行い、必要な追加分析を最大3回実行する。最後にカーネル再起動後、全コードセルを上からJupyter MCPで実行して根拠と図表を監査する。

結論は観察データ内の関連に限定する。糖尿病の原因・発症リスク・介入効果を推定したとは述べない。群別の補完はOutcomeを使うため行わない。予測性能評価や機械学習モデルの選択は本分析の目的ではない。

## 実行・保存経路
Notebookの作成・セル構築はproject_manager.ensure_notebook/enqueue_write、実行・出力保存はJupyter MCPのexecute_cellで行う。端末・subprocess・外部スクリプト・作業エージェントは利用しない。mcp_gateway.run_and_recordはPython側へMCPクライアントを注入する必要があるが、このCLIのツール接続はPythonオブジェクトとして提供されないため直接利用できない。偽のクライアントやexecで代替せず、実際のJupyter MCPで全コードを実行する。制御Notebookは構築・監査のみを担当する。
'''
lifecycle.mark_write_start(run_id)
try:
    def initialize(nb):
        if nb.cells:
            raise RuntimeError('保存先が既存Notebookのため上書きを中止')
        nb.cells = [nbformat.v4.new_markdown_cell(intro), nbformat.v4.new_code_cell(setup_source, id='setup'), nbformat.v4.new_code_cell(acquisition_source, id='acquisition')]
        nb.metadata['run_id'] = run_id
        nb.metadata['kernelspec'] = {'display_name':'Python 3','language':'python','name':'python3'}
    pm.enqueue_write(handle, initialize)
finally:
    lifecycle.mark_write_end(run_id)
print('取得セルまで構築しました')

取得セルまで構築しました


In [5]:
acquisition_source = acquisition_source.replace("            if requested_slug not in search_refs:\n                raise LookupError('指定した公開データセットが検索結果に存在しない')\n", "            # 検索先頭ページにないことは取得不能の証拠ではないため指定slugも直接照会する。\n")
def revise_acquisition(nb):
    cell = next(c for c in nb.cells if c.get('id') == 'acquisition')
    cell.source = acquisition_source
    cell.outputs = []
    cell.execution_count = None
write_notebook(revise_acquisition)
print('検索先頭ページだけで取得不能と判定せず、指定slugのファイルAPIを照会するよう修正')

検索先頭ページだけで取得不能と判定せず、指定slugのファイルAPIを照会するよう修正


In [7]:
old_listing = "            listing = api.dataset_list_files(requested_slug)\n            file_names = [item.name for item in listing.files]\n            if filename not in file_names:\n                raise FileNotFoundError('指定CSVがデータセットに存在しない')\n"
new_listing = """            candidate_failures = []
            selected_slug = None
            for candidate_slug in [requested_slug, 'ahmedgamelwaly/pima-indians-diabetes', 'harsh146/pima-indians-diabetes']:
                try:
                    listing = api.dataset_list_files(candidate_slug)
                    file_names = [item.name for item in listing.files]
                    if filename not in file_names:
                        raise FileNotFoundError('対応CSVがファイル一覧に存在しない')
                    selected_slug = candidate_slug
                    break
                except Exception as candidate_error:
                    response = getattr(candidate_error, 'response', None)
                    http_status = getattr(response, 'status_code', None)
                    candidate_failures.append({'slug': candidate_slug, 'stage': 'dataset_list_files',
                        'error_type': type(candidate_error).__name__, 'http_status': http_status if isinstance(http_status, int) else None})
            if selected_slug is None:
                raise LookupError('指定版と対応する検索候補2件のファイルAPI照会がいずれも失敗')
"""
acquisition_source = acquisition_source.replace(old_listing, new_listing).replace("api.dataset_download_file(requested_slug, filename", "api.dataset_download_file(selected_slug, filename").replace("source_url = 'https://www.kaggle.com/datasets/' + requested_slug", "source_url = 'https://www.kaggle.com/datasets/' + selected_slug").replace("'owner_dataset_slug': requested_slug if source_kind == 'kaggle' else None", "'owner_dataset_slug': selected_slug if source_kind == 'kaggle' else None").replace("'search_results': search_refs,", "'candidate_failures': locals().get('candidate_failures', []), 'search_results': search_refs,")
write_notebook(revise_acquisition)
print('対応する検索候補2件もファイルAPIで確認する取得手順を保存')

対応する検索候補2件もファイルAPIで確認する取得手順を保存


In [8]:
definition_source = r'''# データ定義manifest・意味的異常検査・欠損処理
with phase('data-definition-and-cleaning'):
    features = [c for c in raw.columns if c != 'Outcome']
    zero_missing = ['Glucose','BloodPressure','SkinThickness','Insulin','BMI']
    definitions = {
      'Pregnancies': ('妊娠回数', '回'), 'Glucose': ('グルコース測定値（負荷後測定と推定、時点は未確認）','mg/dLと推定'),
      'BloodPressure': ('拡張期血圧と推定','mmHgと推定'), 'SkinThickness': ('上腕三頭筋皮下厚と推定','mmと推定'),
      'Insulin': ('インスリン測定値（測定時点は未確認）','µU/mLと推定'), 'BMI': ('体格指数と推定','kg/m²と推定'),
      'DiabetesPedigreeFunction': ('糖尿病家族歴に関する指標と推定（算式未確認）','指標値'),
      'Age': ('年齢と推定','年と推定'), 'Outcome': ('糖尿病に関する二値ラベル、1を陽性と推定','0/1')}
    FV = data_definition.FieldValue
    variables = {col: {'definition': FV(text, 'inferred', provenance['source_url']),
                       'unit': FV(unit, 'inferred', '列名と既知のPimaデータ辞書を参考。ただし掲載版の辞書は検証できていない')} for col,(text,unit) in definitions.items()}
    variables['Outcome']['diagnostic_criteria'] = FV(None, 'unknown', 'CSVに診断基準が含まれない')
    variables['Outcome']['measurement_timing'] = FV(None, 'unknown', '説明変数との時間順序を確認できない')
    definition_manifest = data_definition.build_manifest(
      source={'url': FV(provenance['source_url'],'verified','取得記録'),
              'sha256': FV(sha256,'verified','取得ファイル計算'),
              'license': FV(None,'unknown','CSV単体からライセンスを確認できない')},
      dataset_scope={'population': FV('Pima集団の成人女性（21歳以上）とされるデータ。今回のCSVだけでは属性を検証できない','inferred', 'Kaggle参照先・列名'),
                     'sampling_frame': FV(None,'unknown','標本抽出方法は未確認'),
                     'row_count': FV(len(raw),'verified','読込結果')},
      variables=variables, transformations=('5列のゼロをNaNへ変換。Pregnanciesの0は保持。正の極端値は主解析で保持。',))
    print('データ定義manifest:', json.dumps(asdict(definition_manifest), ensure_ascii=False, default=str))
    print('unknown fields:', [(k,asdict(v)) for k,v in definition_manifest.unresolved_fields()])
    inferred_fields = [f'variables.{col}.{field}' for col,fields in variables.items() for field,value in fields.items() if value.status == 'inferred']
    print('inferred fields（別途列挙）:', inferred_fields)
    broad_bounds = {'Pregnancies':(0,30),'Glucose':(1,600),'BloodPressure':(1,250),
        'SkinThickness':(1,150),'Insulin':(1,3000),'BMI':(1,100),'DiabetesPedigreeFunction':(0,10),'Age':(21,120)}
    schema = {col:{'min':lo,'max':hi,'not_null':True} for col,(lo,hi) in broad_bounds.items()}
    schema['Outcome'] = {'allowed':[0,1],'not_null':True}
    semantic_issues = data_quality.detect_anomalies(raw, schema)
    display(pd.DataFrame([asdict(x) for x in semantic_issues]).drop(columns='row_indices', errors='ignore'))
    print('整数性違反:', {c:int((raw[c] % 1 != 0).sum()) for c in ['Pregnancies','Age','Outcome']})
    print('元のNA数:', int(raw.isna().sum().sum()), '重複行数:', int(raw.duplicated().sum()))
    duplicate_report = cleaning.clean_dataset(raw, operation='drop_duplicates')
    print('重複除去影響:', duplicate_report)
    clean = raw.drop_duplicates().copy()
    clean[zero_missing] = clean[zero_missing].mask(clean[zero_missing].eq(0))
    assert len(clean) == len(raw), '重複が存在する場合は解析単位を再検討する'
    assert clean['Outcome'].isin([0,1]).all()
    print('ゼロ→欠損の列別件数:', clean[zero_missing].isna().sum().to_dict())
    missing_table = pd.DataFrame({'missing_n': clean[features].isna().sum(), 'missing_pct': clean[features].isna().mean()*100})
    for outcome in [0,1]:
        group = clean.loc[clean.Outcome == outcome, features]
        missing_table[f'Outcome{outcome}_n'] = group.isna().sum()
        missing_table[f'Outcome{outcome}_pct'] = group.isna().mean()*100
    display(missing_table.round(3))
    explore_report = eda.explore(clean)
    print('EDA型:', clean.dtypes.to_dict())
    display(clean.describe().T)
    print('EDAモジュール出力:', explore_report)
    assumptions_manifest = analysis_assumptions.AnalysisAssumptionManifest(
      analysis_scope={'target':'この標本内のOutcomeと各特徴の関連','generalization':'外部集団・将来発症への外挿はしない'},
      causal_scope='associational', sampling={'n':len(clean),'seed':20261002},
      assumptions=(
        analysis_assumptions.Assumption('zero-as-missing','5列の0は欠測コードとみなす','assumed',impact_if_false='測定定義や関連の推定が変わる',conclusion_critical=True),
        analysis_assumptions.Assumption('independent-rows','各行は独立した観測と仮定。個人IDがなく再測定は検証できない','assumed',impact_if_false='CI・検定の精度が過大になる',conclusion_critical=True),
        analysis_assumptions.Assumption('outcome-semantics','Outcome=1を糖尿病陽性として解釈する。診断基準と時間順序は不明','assumed',impact_if_false='臨床的解釈が変わる',conclusion_critical=True),
        analysis_assumptions.Assumption('missingness','完全例・中央値補完で欠損バイアスを解消できるとは仮定しない','rejected',impact_if_false='欠損が非ランダムなら関連が偏る'),
        analysis_assumptions.Assumption('causality','観察的関連のみ。因果識別の設計はない','verified',impact_if_false='因果への誤解を招く')))
    print('分析前提manifest:', json.dumps(asdict(assumptions_manifest),ensure_ascii=False))
    assumption_findings = analysis_assumptions.check_manifest(assumptions_manifest)
    for finding in assumption_findings:
        print('前提の未解決警告:', asdict(finding))
    print('適用不可: 独立した参照データは未提供。validate_anomalies/compare_datasetsは同一由来の複製を独立検証と誤認しないため使用しない。z-scoreによる一律削除も歪んだ分布・真の高値を誤削除するため行わない。')
'''
comparison_source = r'''# Outcome別比較、点双列相関、効果量・95%CI（観測値のみ）
with phase('univariate-association'):
    from statsmodels.stats.multitest import multipletests
    group_counts = clean.Outcome.value_counts().sort_index()
    summary = {'n': len(clean), 'positive': int(group_counts[1]), 'negative': int(group_counts[0]), 'positive_pct': clean.Outcome.mean()*100}
    display(Markdown('### 標本構成\n```json\n'+json.dumps(summary)+'\n```'))
    rng = np.random.default_rng(20261002)
    rows = []
    def hedges_g(a,b):
        pooled = np.sqrt(((len(a)-1)*np.var(a,ddof=1)+(len(b)-1)*np.var(b,ddof=1))/(len(a)+len(b)-2))
        return (1-3/(4*(len(a)+len(b))-9))*(np.mean(b)-np.mean(a))/pooled
    for column in features:
        available = clean[[column,'Outcome']].dropna()
        a = available.loc[available.Outcome==0,column].to_numpy()
        b = available.loc[available.Outcome==1,column].to_numpy()
        stat = stats_analysis.correlation(available, column, 'Outcome', language='ja')
        boot_a = rng.choice(a, size=(2000,len(a)),replace=True)
        boot_b = rng.choice(b, size=(2000,len(b)),replace=True)
        pooled_boot = np.sqrt(((len(a)-1)*boot_a.var(axis=1,ddof=1)+(len(b)-1)*boot_b.var(axis=1,ddof=1))/(len(a)+len(b)-2))
        boot_g = (1-3/(4*(len(a)+len(b))-9))*(boot_b.mean(axis=1)-boot_a.mean(axis=1))/pooled_boot
        lo,hi = np.quantile(boot_g,[.025,.975])
        rows.append({'feature':column,'n0':len(a),'n1':len(b),'median0':np.median(a),'median1':np.median(b),
            'Q1_0':np.quantile(a,.25),'Q3_0':np.quantile(a,.75),'Q1_1':np.quantile(b,.25),'Q3_1':np.quantile(b,.75),
            'mean0':a.mean(),'mean1':b.mean(),'hedges_g':hedges_g(a,b),'g_ci_low':lo,'g_ci_high':hi,
            'point_biserial_r':stat.statistic,'p':stat.p_value,'interpretation':stat.interpretation})
    association = pd.DataFrame(rows).set_index('feature')
    association['q_BH'] = multipletests(association['p'],method='fdr_bh')[1]
    display(association.drop(columns='interpretation').round(6))
    display(Markdown('### 相関の解釈\n'+ '\n\n'.join(f'**{column}**: {row.interpretation} 二値OutcomeとのPearson相関は点双列相関に相当し、因果関係は示さない。' for column,row in association.iterrows())))
    display(Markdown('### 引用用出力\n```json\n'+json.dumps({'Glucose_r':round(association.loc['Glucose','point_biserial_r'],6),'Glucose_g':round(association.loc['Glucose','hedges_g'],6),'BMI_r':round(association.loc['BMI','point_biserial_r'],6),'Age_r':round(association.loc['Age','point_biserial_r'],6),'Insulin_missing_pct':round(missing_table.loc['Insulin','missing_pct'],6)},ensure_ascii=False)+'\n```'))
'''
charts_source = r'''# 比較図：欠損率、群別分布、標準化群間差。各図はNotebookにPNGを保存。
with phase('comparison-visualization'):
    import matplotlib.pyplot as plt
    import warnings
    def show_png(png):
        output = visualization.build_image_output(png)
        display(output['data'], raw=True)
    def show_figure(fig):
        with warnings.catch_warnings(record=True) as messages:
            warnings.simplefilter('always')
            buf = io.BytesIO()
            fig.savefig(buf,format='png',dpi=130,bbox_inches='tight')
        plt.close(fig)
        missing_glyph_messages = [str(w.message) for w in messages if 'Glyph' in str(w.message) and 'missing' in str(w.message)]
        if missing_glyph_messages:
            raise RuntimeError('文字欠落を検出: '+repr(missing_glyph_messages))
        show_png(buf.getvalue())
    missing_plot = missing_table[['Outcome0_pct','Outcome1_pct']].rename(columns={'Outcome0_pct':'Outcome=0','Outcome1_pct':'Outcome=1'}).reset_index(names='Feature')
    with warnings.catch_warnings(record=True) as messages:
        warnings.simplefilter('always')
        png = visualization.render_chart(missing_plot,kind='bar',x='Feature',y=['Outcome=0','Outcome=1'],title='Outcome別の欠損率（不自然なゼロを欠損扱い）',xlabel='特徴',ylabel='欠損率 (%)')
    assert not any('Glyph' in str(w.message) and 'missing' in str(w.message) for w in messages)
    show_png(png)
    fig,axes = plt.subplots(2,4,figsize=(15,8))
    for ax,column in zip(axes.flat,features):
        samples = [clean.loc[clean.Outcome==outcome,column].dropna() for outcome in [0,1]]
        ax.boxplot(samples,tick_labels=['0','1'],showfliers=True)
        ax.set_title(column)
        ax.set_xlabel('Outcome (0 / 1)')
        ax.set_ylabel('Observed value (native scale)')
        ax.text(.02,.97,f'n0={len(samples[0])}, n1={len(samples[1])}',transform=ax.transAxes,va='top',fontsize=8)
    fig.suptitle('Observed distributions by Outcome: medians, IQR, whiskers and outliers')
    fig.tight_layout()
    show_figure(fig)
    ordered = association.sort_values('hedges_g')
    fig,ax = plt.subplots(figsize=(8,5))
    ax.errorbar(ordered.hedges_g,ordered.index,xerr=[ordered.hedges_g-ordered.g_ci_low,ordered.g_ci_high-ordered.hedges_g],fmt='o',label='Observed data; bootstrap 95% CI')
    ax.axvline(0,color='gray',linestyle='--')
    ax.set(title='Standardized differences: Outcome=1 minus Outcome=0',xlabel='Hedges g (positive = higher in Outcome=1)',ylabel='Feature')
    ax.legend(loc='lower right',fontsize=8)
    show_figure(fig)
'''
model_source = r'''# 調整ロジスティック回帰と欠損処理・極端値処理の感度分析
with phase('adjusted-association-and-sensitivity'):
    scale = clean[features].std(ddof=1)
    center = clean[features].mean()
    model_cache = {}
    def fit_model(strategy='median',tail='keep',indicators=True):
        key = (strategy,tail,indicators)
        if key in model_cache:
            return model_cache[key]
        work = clean.copy()
        if tail == 'winsor':
            for column in features:
                lo,hi = work[column].quantile([.01,.99])
                work[column] = work[column].clip(lo,hi)
        elif tail != 'keep':
            raise ValueError(tail)
        if strategy == 'complete':
            work = work.dropna(subset=features)
            x = (work[features]-center)/scale
        elif strategy == 'median':
            x = (work[features].fillna(work[features].median())-center)/scale
            if indicators:
                for column in zero_missing:
                    x[column+'_missing'] = work[column].isna().astype(int)
        else:
            raise ValueError(strategy)
        fitted = sm.GLM(work.Outcome,sm.add_constant(x),family=sm.families.Binomial()).fit()
        assert fitted.converged
        model_cache[key] = (fitted,len(work))
        return fitted,len(work)
    main_model,main_n = fit_model()
    ci = main_model.conf_int()
    adjusted = pd.DataFrame({'beta':main_model.params,'adjusted_OR_per_SD':np.exp(main_model.params),
        'CI_low':np.exp(ci[0]),'CI_high':np.exp(ci[1]),'p':main_model.pvalues}).drop('const')
    display(adjusted.round(6))
    print('主解析n:',main_n,'全体の観測SDを共通基準に標準化。補完はOutcome非参照。欠損指示変数5列を追加。')
    print('注: 補完後の通常CIは補完値の不確実性を含まない。調整ORは関連であり因果効果・リスク比ではない。')
    sensitivity_plan = sensitivity.SensitivityPlan(parameter_grid={'strategy':['median','complete'],'tail':['keep','winsor']},max_runs=4)
    sensitivity_reports = {}
    sensitivity_rows = []
    for column in features:
        report = sensitivity.run_sensitivity(sensitivity_plan,lambda strategy,tail: fit_model(strategy,tail)[0].params[column],stability_tolerance=.20)
        sensitivity_reports[column] = report
        for result in report.results:
            fitted,n = fit_model(**result.specification)
            bound = fitted.conf_int().loc[column]
            sensitivity_rows.append({'feature':column,**result.specification,'n':n,'beta':result.value,
                'OR':np.exp(result.value),'CI_low':np.exp(bound.iloc[0]),'CI_high':np.exp(bound.iloc[1])})
    sensitivity_table = pd.DataFrame(sensitivity_rows)
    display(sensitivity_table.round(6))
    stability_table = pd.DataFrame([{'feature':column,'stable':report.stable,'max_relative_deviation':report.max_relative_deviation,
        'same_direction':all(result.value>0 for result in report.results) or all(result.value<0 for result in report.results)} for column,report in sensitivity_reports.items()]).set_index('feature')
    display(stability_table)
    print('stableは係数の相対変動20%以内を意味し、因果妥当性・欠損機序を保証しない。')
    display(Markdown('### 調整比較の引用用出力\n```json\n'+json.dumps({'Glucose_adjusted_OR':round(adjusted.loc['Glucose','adjusted_OR_per_SD'],6), 'BMI_adjusted_OR':round(adjusted.loc['BMI','adjusted_OR_per_SD'],6), 'Glucose_stable':bool(stability_table.loc['Glucose','stable']), 'Glucose_max_relative_deviation':round(stability_table.loc['Glucose','max_relative_deviation'],6)},ensure_ascii=False)+'\n```'))
'''
policy = '''## データ処理・統計の事前方針
Glucose・BloodPressure・SkinThickness・Insulin・BMIの0は生理的に不自然な値と判断してNaNへ変換する。妊娠回数0とOutcome=0は有効値として残す。意味的検査の広い上下限は診断閾値ではなく明らかなコード異常を見つけるための便宜的範囲。正の高値は誤入力と断定せず主解析で保持し、1/99パーセンタイルへのWinsor化を感度分析に限る。

単変量比較は列ごとの観測値で行い、nを表示する。Hedges gはOutcome=1−0の標準化平均差、CIは群別2000回bootstrap（seed固定）、点双列相関のp値は8特徴のBH補正を併記する。全列共通の標本ではないので順位は探索的。調整分析は全特徴と5列の欠損指示変数、Outcome非参照の全体中央値補完を主解析に用いる。完全例およびWinsor化と比較し、同じ観測SDに標準化して係数を比較する。ORをリスク比と呼ばない。単一補完のCIは補完不確実性を反映しない。

単位・対象母集団・ラベル定義は取得CSVだけから検証できないものをinferred/unknownとして表示する。診断基準と時間順序が不明なため、特にGlucoseとの強い関連にはラベル定義との近接性があり得る。独立した参照データがないためデータセット間の検証は適用不可。モデルの形式や未測定交絡が正しいと保証せず、因果識別・将来予測の検証・外的妥当性検証は行わない。
'''
def append_initial(nb):
    nb.cells += [nbformat.v4.new_markdown_cell(policy),
        nbformat.v4.new_code_cell(definition_source,id='definition-cleaning'),
        nbformat.v4.new_code_cell(comparison_source,id='comparison'),
        nbformat.v4.new_code_cell(charts_source,id='charts',metadata={'chart':{'title':'Outcome別欠損率・群別分布・Hedges g','xlabel':'特徴 / Outcome / 標準化群間差','ylabel':'欠損率 / 観測値 / 特徴','legend':'Outcome=0,1 / bootstrap 95% CI','missing_glyphs':[]}}),
        nbformat.v4.new_code_cell(model_source,id='adjusted-sensitivity')]
write_notebook(append_initial)
print('データ定義・比較・図表・調整分析・感度分析セルを保存')

データ定義・比較・図表・調整分析・感度分析セルを保存


In [15]:
with phase('initial-review-and-iteration-1-plan'):
    initial_audit = notebook_audit.audit_notebook(handle.notebook_path, visual_audit=True)
    print('初回監査（目視でラベル切れを発見）:', asdict(initial_audit), 'ok=', initial_audit.ok)
    initial_audit_log = {'report':asdict(initial_audit),'ok':initial_audit.ok,'manual_visual_finding':'render_chartの欠損率棒グラフは長いx目盛りとxlabelが下端で切れる。visual_findingsは空。'}
    (data_dir / 'initial-visual-audit.json').write_text(json.dumps(initial_audit_log,ensure_ascii=False,indent=2),encoding='utf-8')
    snapshot = nbformat.read(handle.notebook_path,as_version=4)
    evidence_cell = next(c for c in snapshot.cells if c.get('id')=='comparison')
    evidence_output = '\n'.join(str(value) for out in evidence_cell.outputs for value in out.get('data',{}).values())
    cited = insight_engine.extract_cited_value({'output':evidence_output}, r'"Glucose_r": ([0-9.]+)')
    lifecycle.mark_write_start(run_id)
    try:
        insight_engine.record_insight(handle, '初回Insight：Glucoseはこの標本でOutcomeとの点双列相関が最も大きく、r='+cited+'。BMI・Insulinも正の単変量関連があるが、Insulinの欠損が多いため独立した重要性を断定しない。', evidence_cell.execution_count, cited,'associational',language='ja')
    finally:
        lifecycle.mark_write_end(run_id)
    # 長いラベルの切れは最終成果物で解消する。render_chartも別図で実利用する。
    chart_start = charts_source.index('    missing_plot =')
    chart_end = charts_source.index('    fig,axes =')
    missing_chart_replacement = r'''    missing_plot = missing_table[['Outcome0_pct','Outcome1_pct']].rename(columns={'Outcome0_pct':'Outcome=0','Outcome1_pct':'Outcome=1'})
    fig,ax = plt.subplots(figsize=(9,6))
    missing_plot.plot.barh(ax=ax)
    ax.set(title='Missingness by Outcome: zeros treated as missing',xlabel='Missing values (%)',ylabel='Feature')
    ax.legend(title='Outcome')
    show_figure(fig)
    counts_plot = pd.DataFrame({'Outcome':['0','1'],'Count':[int(group_counts[0]),int(group_counts[1])]})
    with warnings.catch_warnings(record=True) as messages:
        warnings.simplefilter('always')
        png = visualization.render_chart(counts_plot,kind='bar',x='Outcome',y='Count',title='Outcome別の観測数',xlabel='Outcome（0 / 1）',ylabel='観測数')
    assert not any('Glyph' in str(w.message) and 'missing' in str(w.message) for w in messages)
    show_png(png)
'''
    charts_source = charts_source[:chart_start] + missing_chart_replacement + charts_source[chart_end:]
    charts_source = charts_source.replace("ax.boxplot(samples,tick_labels=['0','1'],showfliers=True)", "ax.boxplot(samples,tick_labels=['0','1'],showfliers=True)\n        ax.margins(y=.15)")
    request1 = '欠損処理によってBMIや年齢の調整関連が変わっています。完全例として残る人と除外される人のOutcome比率・主要特徴を比較し、欠損の少ない5特徴だけのモデルと全8特徴モデルを比べて、選択バイアスと共線性が結論を左右するか確認してください。'
    iteration1_source = r'''# 反復1：完全例選択と共線性、欠損の少ない特徴モデル
with phase('iteration-1-selection-and-collinearity'):
    core_features = ['Glucose','BMI','Age','Pregnancies','DiabetesPedigreeFunction']
    complete_flag = clean[features].notna().all(axis=1)
    selection = clean.groupby(complete_flag).agg(n=('Outcome','size'),positive=('Outcome','sum'),positive_rate=('Outcome','mean'),Glucose_median=('Glucose','median'),BMI_median=('BMI','median'),Age_median=('Age','median'))
    selection.index = ['not-complete','complete']
    display(selection)
    selection_fisher = stats.fisher_exact(pd.crosstab(complete_flag, clean.Outcome))
    print('完全例選択とOutcomeのFisher検定:', selection_fisher)
    from statsmodels.stats.outliers_influence import variance_inflation_factor
    vif_x = sm.add_constant((clean[features].fillna(clean[features].median())-center)/scale)
    vif_table = pd.DataFrame({'feature':features,'VIF':[variance_inflation_factor(vif_x.values,i+1) for i in range(len(features))]})
    display(vif_table)
    def fit_core(strategy='median',without_glucose=False):
        columns = [c for c in core_features if not(without_glucose and c=='Glucose')]
        work = clean.dropna(subset=columns) if strategy=='complete' else clean
        x = (work[columns].fillna(work[columns].median())-center[columns])/scale[columns]
        if strategy=='median':
            for column in [c for c in zero_missing if c in columns]:
                x[column+'_missing'] = work[column].isna().astype(int)
        fitted = sm.GLM(work.Outcome,sm.add_constant(x),family=sm.families.Binomial()).fit()
        assert fitted.converged
        return fitted,len(work)
    core_rows = []
    for strategy in ['median','complete']:
        fitted,n = fit_core(strategy)
        for column in core_features:
            ci = fitted.conf_int().loc[column]
            core_rows.append({'strategy':strategy,'feature':column,'n':n,'OR':np.exp(fitted.params[column]),'low':np.exp(ci.iloc[0]),'high':np.exp(ci.iloc[1])})
    core_comparison = pd.DataFrame(core_rows)
    display(core_comparison.round(6))
    display(Markdown('### 反復1の引用用出力\n```json\n'+json.dumps({'complete_n':int(complete_flag.sum()),'complete_positive_rate':round(selection.loc['complete','positive_rate'],6),'excluded_positive_rate':round(selection.loc['not-complete','positive_rate'],6),'max_VIF':round(vif_table.VIF.max(),6)},ensure_ascii=False)+'\n```'))
'''
    def append_iteration1(nb):
        cell = next(c for c in nb.cells if c.get('id')=='charts')
        cell.source = charts_source
        cell.outputs = []
        cell.execution_count = None
        cell.metadata.chart.title = 'Outcome別欠損率・観測数・群別分布・Hedges g'
        nb.cells += [nbformat.v4.new_markdown_cell('## 反復依頼履歴\n\n### 反復1（初回結果の再読後）\n\n**追加依頼原文**\n\n'+request1+'\n\n**選定理由**: BMIの調整係数は完全例処理で約26.5%変動し、年齢はさらに変動した。全8特徴の完全例は392件と約半数になるため、標本選択とモデル調整の影響を分離する必要がある。結果・証拠・限界は直後の実行出力と根拠付きInsightに記録する。'), nbformat.v4.new_code_cell(iteration1_source,id='iteration-1')]
    write_notebook(append_iteration1)
    print('反復1依頼原文と実行セル、図表修正版を保存')

初回監査（目視でラベル切れを発見）: {'path': '/home/nahisaho/GitHub/jupytermind/benchmark/projects/kaggle-v020-kaggle-exp-09-diabetes/notebooks/kaggle-v020-kaggle-exp-09-diabetes.ipynb', 'nbformat_valid': True, 'code_cell_count': 6, 'executed_code_cell_count': 6, 'unexecuted_cell_indices': (), 'error_cell_indices': (), 'chart_cell_indices': (6,), 'insight_cell_count': 0, 'findings': (), 'visual_findings': ()} ok= True
反復1依頼原文と実行セル、図表修正版を保存


In [17]:
with phase('iteration-2-plan'):
    request2 = '年齢は単変量では正の関連がある一方、調整後は不確実です。年齢とGlucoseの非線形性、およびInsulinなどの右裾の長い分布を考慮したモデルで主要な関連を再確認し、単変量の関連と調整後の関連を混同しない結論に更新してください。'
    iteration2_source = r'''# 反復2：非線形性・歪んだ分布に対するモデル感度
with phase('iteration-2-functional-form'):
    import statsmodels.formula.api as smf
    from patsy import bs
    analysis_data = (clean[features].fillna(clean[features].median())-center)/scale
    for column in zero_missing:
        analysis_data[column+'_missing'] = clean[column].isna().astype(int)
    analysis_data['Outcome'] = clean.Outcome
    linear_formula = 'Outcome ~ ' + ' + '.join([*features,*[c+'_missing' for c in zero_missing]])
    linear_formula_model = smf.glm(linear_formula,data=analysis_data,family=sm.families.Binomial()).fit()
    assert np.isclose(linear_formula_model.llf,main_model.llf)
    spline_models = {}
    functional_rows = []
    nonlinear_test_rows = []
    for spline_columns in [['Age'],['Glucose'],['Age','Glucose']]:
        terms = [f'bs({column}, df=4, degree=3, include_intercept=False)' if column in spline_columns else column for column in features]
        formula = 'Outcome ~ ' + ' + '.join(terms+[c+'_missing' for c in zero_missing])
        fitted = smf.glm(formula,data=analysis_data,family=sm.families.Binomial()).fit()
        assert fitted.converged
        name = 'spline-'+'-'.join(spline_columns)
        spline_models[name] = fitted
        lr = 2*(fitted.llf-linear_formula_model.llf)
        extra_df = int(fitted.df_model-linear_formula_model.df_model)
        nonlinear_test_rows.append({'model':name,'LR':lr,'extra_df':extra_df,'p_LR':stats.chi2.sf(lr,extra_df),'AIC':fitted.aic})
        for column in features:
            if column not in fitted.params.index:
                continue
            ci = fitted.conf_int().loc[column]
            functional_rows.append({'model':name,'feature':column,'OR_per_observed_SD':np.exp(fitted.params[column]),'low':np.exp(ci.iloc[0]),'high':np.exp(ci.iloc[1])})
    nonlinear_tests = pd.DataFrame(nonlinear_test_rows)
    display(nonlinear_tests)
    display(pd.DataFrame(functional_rows).round(6))
    log_data = analysis_data.drop(columns='Outcome').copy()
    for column in ['Insulin','DiabetesPedigreeFunction']:
        observed_log = np.log1p(clean[column])
        log_data[column] = (observed_log.fillna(observed_log.median())-observed_log.mean())/observed_log.std(ddof=1)
    log_model = sm.GLM(clean.Outcome,sm.add_constant(log_data),family=sm.families.Binomial()).fit()
    assert log_model.converged
    log_rows = []
    for column in features:
        ci = log_model.conf_int().loc[column]
        log_rows.append({'feature':column,'OR':np.exp(log_model.params[column]),'low':np.exp(ci.iloc[0]),'high':np.exp(ci.iloc[1])})
    display(pd.DataFrame(log_rows).round(6))
    print('Insulin/DPFのlogモデルのORはlog1p変換後1SDあたり。他の特徴のSDは主解析と共通。非線形モデルのAge/Glucoseは単一のORで要約しない。')
    age_spline = spline_models['spline-Age']
    profile = pd.DataFrame(0.0,index=range(50),columns=analysis_data.drop(columns='Outcome').columns)
    age_grid = np.linspace(21,70,50)
    profile['Age'] = (age_grid-center['Age'])/scale['Age']
    prediction = age_spline.get_prediction(profile).summary_frame()
    fig,ax = plt.subplots(figsize=(8,5))
    ax.plot(age_grid,prediction['mean'],label='Age spline; other observed features at their means')
    ax.fill_between(age_grid,prediction['mean_ci_lower'],prediction['mean_ci_upper'],alpha=.2,label='Pointwise 95% CI')
    ax.set(title='Model-based Outcome probability across Age (not causal)',xlabel='Age (native scale)',ylabel='Estimated probability of Outcome=1')
    ax.set_ylim(0,1)
    ax.legend(fontsize=8)
    show_figure(fig)
    fig,ax = plt.subplots(figsize=(9,5))
    points = adjusted.loc[features]
    ax.errorbar(points.adjusted_OR_per_SD,points.index,xerr=[points.adjusted_OR_per_SD-points.CI_low,points.CI_high-points.adjusted_OR_per_SD],fmt='o',label='Linear model; median imputation + missing indicators')
    ax.axvline(1,color='gray',linestyle='--')
    ax.set_xscale('log')
    ax.set(title='Adjusted associations: all 8 features',xlabel='Adjusted odds ratio per observed SD (log scale)',ylabel='Feature')
    ax.legend(fontsize=8)
    show_figure(fig)
    functional_summary = {'Age_spline_p_LR':round(float(nonlinear_tests.iloc[0].p_LR),8),
      'Glucose_OR_age_spline':round(float(np.exp(age_spline.params['Glucose'])),6),
      'BMI_OR_age_spline':round(float(np.exp(age_spline.params['BMI'])),6),
      'Glucose_OR_log_model':round(float(np.exp(log_model.params['Glucose'])),6),
      'BMI_OR_log_model':round(float(np.exp(log_model.params['BMI'])),6)}
    display(Markdown('### 反復2の引用用出力\n```json\n'+json.dumps(functional_summary)+'\n```'))
'''
    def append_iteration2(nb):
        nb.cells += [nbformat.v4.new_markdown_cell('## 反復依頼履歴（続き）\n\n### 反復2\n\n**追加依頼原文**\n\n'+request2+'\n\n**選定理由**: 反復1で極端な共線性は確認されなかったが、全8特徴完全例では年齢中央値が27対32と異なり、調整後の年齢の直線係数も不確実。年齢とラベルの関連形状および歪んだ分布が結論を左右する可能性が残る。検定は探索的で多重なモデル探索による過大解釈を避ける。結果・証拠・限界は実行出力と根拠付きInsightに記録する。'), nbformat.v4.new_code_cell(iteration2_source,id='iteration-2',metadata={'chart':{'title':'Age spline / Adjusted associations','xlabel':'Age / Adjusted OR','ylabel':'Estimated probability / Feature','legend':'pointwise 95% CI / median imputation + indicators','missing_glyphs':[]}})]
    write_notebook(append_iteration2)
    print('反復2の原文と実行セルを保存')

反復2の原文と実行セルを保存


In [19]:
with phase('iteration-3-plan'):
    request3 = '年齢を非線形に扱うと妊娠回数の調整関連が弱まり、年齢曲線の高齢端では下降が見られます。年齢帯ごとの標本数とOutcome割合を確認し、スプライン自由度3〜5および60歳以下に限定した感度分析で、Glucose・BMI・家族歴指標・妊娠回数の結論が安定するか調べてください。高齢で糖尿病が予防されると解釈しないでください。'
    iteration3_source = r'''# 反復3：年齢形状の柔軟性と高齢端の少数標本
with phase('iteration-3-age-support-and-sensitivity'):
    from statsmodels.stats.proportion import proportion_confint
    age_bands = pd.cut(clean.Age,bins=[20,29,39,49,59,120],labels=['21-29','30-39','40-49','50-59','60+'])
    age_band_table = clean.groupby(age_bands,observed=True).Outcome.agg(['size','sum','mean']).rename(columns={'size':'n','sum':'positive','mean':'positive_rate'})
    low,high = proportion_confint(age_band_table.positive,age_band_table.n,method='wilson')
    age_band_table['Wilson_low'],age_band_table['Wilson_high'] = low,high
    display(age_band_table.round(6))
    age_cache = {}
    def fit_age_variant(age_max,spline_df):
        key = (age_max,spline_df)
        if key not in age_cache:
            work = analysis_data.loc[clean.Age<=age_max].copy()
            terms = [f'bs(Age, df={spline_df}, degree=3, include_intercept=False)' if column=='Age' else column for column in features]
            formula = 'Outcome ~ ' + ' + '.join(terms+[c+'_missing' for c in zero_missing])
            fitted = smf.glm(formula,data=work,family=sm.families.Binomial()).fit()
            assert fitted.converged
            age_cache[key] = (fitted,len(work))
        return age_cache[key]
    age_plan = sensitivity.SensitivityPlan(parameter_grid={'age_max':[81,60],'spline_df':[4,3,5]},max_runs=6)
    age_reports = {}
    age_rows = []
    for column in ['Glucose','BMI','DiabetesPedigreeFunction','Pregnancies']:
        report = sensitivity.run_sensitivity(age_plan,lambda age_max,spline_df: fit_age_variant(age_max,spline_df)[0].params[column],stability_tolerance=.20)
        age_reports[column] = report
        for result in report.results:
            fitted,n = fit_age_variant(**result.specification)
            ci = fitted.conf_int().loc[column]
            age_rows.append({'feature':column,**result.specification,'n':n,'OR':np.exp(result.value),'low':np.exp(ci.iloc[0]),'high':np.exp(ci.iloc[1])})
    age_sensitivity = pd.DataFrame(age_rows)
    display(age_sensitivity.round(6))
    age_stability = pd.DataFrame([{'feature':column,'stable':report.stable,'max_relative_deviation':report.max_relative_deviation} for column,report in age_reports.items()])
    display(age_stability)
    fig,ax = plt.subplots(figsize=(8,5))
    ax.errorbar(age_band_table.index.astype(str),age_band_table.positive_rate,yerr=[age_band_table.positive_rate-age_band_table.Wilson_low,age_band_table.Wilson_high-age_band_table.positive_rate],fmt='o',capsize=5,label='Observed proportion; Wilson 95% CI')
    for pos,(_,row) in enumerate(age_band_table.iterrows()):
        ax.text(pos,min(row.Wilson_high+.05,.96),f'n={int(row.n)}',ha='center')
    ax.set(title='Outcome=1 proportions by Age band: descriptive only',xlabel='Age band (native scale)',ylabel='Proportion of Outcome=1')
    ax.set_ylim(0,1)
    ax.legend(fontsize=8)
    show_figure(fig)
    display(Markdown('### 反復3の引用用出力\n```json\n'+json.dumps({'Age60plus_n':int(age_band_table.loc['60+','n']), 'Glucose_OR_age_variants_min':round(float(age_sensitivity.loc[age_sensitivity.feature=='Glucose','OR'].min()),6),'Glucose_OR_age_variants_max':round(float(age_sensitivity.loc[age_sensitivity.feature=='Glucose','OR'].max()),6),'BMI_OR_age_variants_min':round(float(age_sensitivity.loc[age_sensitivity.feature=='BMI','OR'].min()),6)},ensure_ascii=False)+'\n```'))
'''
    def append_iteration3(nb):
        nb.cells += [nbformat.v4.new_markdown_cell('## 反復依頼履歴（続き）\n\n### 反復3\n\n**追加依頼原文**\n\n'+request3+'\n\n**選定理由**: 反復2では年齢スプラインの追加で適合が改善し、妊娠回数のORが約1.52から1.19に弱まった。高齢端の曲線を一般化すると誤解につながるため、観測範囲の支持と形状依存性を確認する。最大3回の最終反復。結果・証拠・限界は実行出力と根拠付きInsightに記録する。'),nbformat.v4.new_code_cell(iteration3_source,id='iteration-3',metadata={'chart':{'title':'Outcome proportions by Age band','xlabel':'Age band','ylabel':'Proportion Outcome=1','legend':'Wilson 95% CI','missing_glyphs':[]}})]
    write_notebook(append_iteration3)
    print('反復3の原文と実行セルを保存')

反復3の原文と実行セルを保存


In [21]:
with phase('finalization-plan-and-replay-preparation'):
    # リプレイ比較用の初回数値を固定し、データ/解析の変化を明示的に検出する。
    first_fingerprint = {'sha256':sha256,'n':len(clean),'positive':int(clean.Outcome.sum()),
        'Glucose_r':float(association.loc['Glucose','point_biserial_r']),
        'BMI_r':float(association.loc['BMI','point_biserial_r']),
        'Glucose_beta':float(main_model.params['Glucose']), 'BMI_beta':float(main_model.params['BMI']),
        'Age_spline_llf':float(spline_models['spline-Age'].llf),
        'Glucose_age_min_OR':float(age_sensitivity.loc[age_sensitivity.feature=='Glucose','OR'].min())}
    lifecycle.mark_write_start(run_id)
    try:
        (data_dir/'first-analysis-fingerprint.json').write_text(json.dumps(first_fingerprint,indent=2),encoding='utf-8')
        (data_dir/'initial-phase-log.json').write_text(json.dumps(phase_log,ensure_ascii=False,indent=2),encoding='utf-8')
    finally:
        lifecycle.mark_write_end(run_id)
    acquisition_source = acquisition_source.replace("            api.dataset_download_file(selected_slug, filename", "            if downloaded_path.exists():\n                downloaded_path.unlink()\n            api.dataset_download_file(selected_slug, filename")
    acquisition_source = acquisition_source.replace("print('前回取得SHAとの一致:', prior_provenance['sha256'] == sha256)", "print('前回取得SHAとの一致:', prior_provenance['sha256'] == sha256)\n        assert prior_provenance['sha256'] == sha256, '再取得でファイルSHAが変化したため再現性の判定を中止'")
    acquisition_source = acquisition_source.replace("'identity_warning': None if failure is None else '公開CSVとKaggle掲載版の完全な同一性は保証されない'", "'identity_warning': ('対応する別ownerのKaggle公開版。参照指定uciml版との完全な同一性は未検証' if selected_slug != requested_slug else None) if failure is None else '公開CSVとKaggle掲載版の完全な同一性は保証されない'")
    definition_source = definition_source.replace("print('重複除去影響:', duplicate_report)", "print('重複除去影響:', {'rows_before':duplicate_report.rows_before,'rows_after':duplicate_report.rows_after,'rows_removed':duplicate_report.rows_removed,'columns_affected':duplicate_report.columns_affected})").replace("clean = raw.drop_duplicates().copy()", "clean = duplicate_report.dataframe.copy()").replace("print('EDAモジュール出力:', explore_report)", "print('EDA欠損サマリ:', explore_report.missing_summary)\n    print('EDA非欠損件数:', explore_report.non_null_counts)")
    comparison_source = comparison_source.replace("display(association.drop(columns='interpretation').round(6))", "association_display = association.drop(columns='interpretation').round(6)\n    for p_column in ['p','q_BH']:\n        association_display[p_column] = association[p_column].map(lambda value: f'{value:.3e}')\n    display(association_display)")
    final_summary_source = r'''# 日本語の最終結論・反復報告・再実行時の数値照合
with phase('final-summary-and-replay-check'):
    current_fingerprint = {'sha256':sha256,'n':len(clean),'positive':int(clean.Outcome.sum()),
        'Glucose_r':float(association.loc['Glucose','point_biserial_r']),
        'BMI_r':float(association.loc['BMI','point_biserial_r']),
        'Glucose_beta':float(main_model.params['Glucose']), 'BMI_beta':float(main_model.params['BMI']),
        'Age_spline_llf':float(spline_models['spline-Age'].llf),
        'Glucose_age_min_OR':float(age_sensitivity.loc[age_sensitivity.feature=='Glucose','OR'].min())}
    first_fingerprint = json.loads((data_dir/'first-analysis-fingerprint.json').read_text(encoding='utf-8'))
    for key,value in current_fingerprint.items():
        if isinstance(value,(float,int)):
            assert np.isclose(value,first_fingerprint[key],rtol=1e-9,atol=1e-10), f'再実行値が不一致: {key}'
        else:
            assert value == first_fingerprint[key], f'再実行ファイルが不一致: {key}'
    print('初回解析と再実行のSHA・9項目の数値照合: 一致')
    print('照合記録:', json.dumps(current_fingerprint,ensure_ascii=False))
    interpretation_text = f''' + "'''" + r'''## 日本語の結論
この標本768件（Outcome=1は268件、34.9%）では、**Glucoseが最も明瞭な正の関連**を示した。ゼロを欠損にした観測値の点双列相関は{association.loc['Glucose','point_biserial_r']:.3f}、Hedges gは{association.loc['Glucose','hedges_g']:.2f}（bootstrap 95%CI {association.loc['Glucose','g_ci_low']:.2f}–{association.loc['Glucose','g_ci_high']:.2f}）。BMIと家族歴指標も、調整・年齢形状の検討を通じて正の関連が残った。

主解析の調整OR（観測値1SDあたり）はGlucose {adjusted.loc['Glucose','adjusted_OR_per_SD']:.2f}（95%CI {adjusted.loc['Glucose','CI_low']:.2f}–{adjusted.loc['Glucose','CI_high']:.2f}）、BMI {adjusted.loc['BMI','adjusted_OR_per_SD']:.2f}（{adjusted.loc['BMI','CI_low']:.2f}–{adjusted.loc['BMI','CI_high']:.2f}）、家族歴指標 {adjusted.loc['DiabetesPedigreeFunction','adjusted_OR_per_SD']:.2f}（{adjusted.loc['DiabetesPedigreeFunction','CI_low']:.2f}–{adjusted.loc['DiabetesPedigreeFunction','CI_high']:.2f}）。Glucoseの係数は欠損処理・Winsor化4仕様で最大相対変動{sensitivity_reports['Glucose'].max_relative_deviation:.1%}、stable={sensitivity_reports['Glucose'].stable}。BMIは同検討で{ sensitivity_reports['BMI'].max_relative_deviation:.1%}変動しstable=Falseのため、方向は一貫しても大きさまで頑健とは言わない。

年齢の単変量関連を単調な独立効果と解釈しない。年齢を柔軟に扱うと妊娠回数のORは{np.exp(spline_models['spline-Age'].params['Pregnancies']):.2f}となりCIが1を含んだ。年齢モデルの6仕様でも妊娠回数は不安定。Insulin・皮下厚・血圧は単変量で正の関連があっても、他特徴と同時に調整した関連は不確実であり、「関係がない」とも「独立した危険因子」とも断定しない。

**これは相関・条件付き関連であり、因果ではない。** Glucoseの測定とOutcome定義の時間順序・診断基準を検証できず、ラベルとの定義上の近さもあり得る。年齢曲線の高齢端の下降は少数標本・選択・未測定要因・関数形の影響を受け得るため予防効果を意味しない。一般集団や将来の糖尿病発症、介入による改善への外挿は行わない。

## 反復結果・根拠・残る限界
1. **反復1（iteration-1）**: 全8特徴の完全例392件と除外376件ではOutcome=1割合が33.2%対36.7%、年齢中央値が27対32。Fisher検定の非有意は欠損がランダムという証明ではない。VIF最大{vif_table.VIF.max():.2f}と極端な共線性は確認されなかった。5特徴の完全例752件と中央値補完768件ではGlucose/BMIの関連は近い。限界: 欠損機序と個人ID不在による独立性は未検証。
2. **反復2（iteration-2）**: 年齢スプラインで適合改善（探索的LR p={functional_summary['Age_spline_p_LR']:.3g}）。妊娠回数の独立した関連の断定を取り下げる。log変換モデルでもGlucose/BMIは正の関連。限界: 非線形モデルの事後選択、単一補完の不確実性、曲線CIは点ごとのCIで同時信頼帯ではない。
3. **反復3（iteration-3）**: 60歳以上は32件。年齢スプライン自由度3–5、全範囲/60歳以下でGlucoseのORは{age_sensitivity.loc[age_sensitivity.feature=='Glucose','OR'].min():.2f}–{age_sensitivity.loc[age_sensitivity.feature=='Glucose','OR'].max():.2f}、BMI/家族歴指標もこの6仕様ではstable=True。限界: 年齢制限は対象を変える。高齢端の下降の機序はこのデータから識別できない。

**反復停止理由**: 3回の上限に達し、主要な関連の方向とモデル依存な結論を区別できた。残る重要点（診断定義、欠損機序、外的妥当性、未測定交絡）は同じCSVの追加モデル探索では解決できない。単位・母集団のinferred/unknownおよび前提manifestの3警告は解決済み扱いにしない。
''' + "'''" + r'''
    display(Markdown(interpretation_text))
    display(Markdown('### 最終照合値\n```json\n'+json.dumps({'Glucose_r':round(association.loc['Glucose','point_biserial_r'],6), 'Glucose_OR':round(adjusted.loc['Glucose','adjusted_OR_per_SD'],6), 'BMI_OR':round(adjusted.loc['BMI','adjusted_OR_per_SD'],6),'DPF_OR':round(adjusted.loc['DiabetesPedigreeFunction','adjusted_OR_per_SD'],6),'Pregnancies_OR_age_spline':round(float(np.exp(spline_models['spline-Age'].params['Pregnancies'])),6),'Insulin_missing_pct':round(missing_table.loc['Insulin','missing_pct'],6)},ensure_ascii=False)+'\n```'))
    lifecycle.mark_write_start(run_id)
    try:
        manifests = {'provenance':provenance,'definition':asdict(definition_manifest),'assumptions':asdict(assumptions_manifest),
           'assumption_findings':[asdict(f) for f in assumption_findings],
           'sensitivity':{column:asdict(report) for column,report in sensitivity_reports.items()},
           'age_sensitivity':{column:asdict(report) for column,report in age_reports.items()},
           'fingerprint':current_fingerprint}
        (data_dir/'analysis-manifests.json').write_text(json.dumps(manifests,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
    finally:
        lifecycle.mark_write_end(run_id)
'''
    audit_source = r'''# Notebook監査とライフサイクルの終了
with phase('final-notebook-audit'):
    audit_report = notebook_audit.audit_notebook(handle.notebook_path,visual_audit=True)
    print('Notebook監査（このセルの実行途中）:', json.dumps(asdict(audit_report),ensure_ascii=False))
    print('audit_ok:',audit_report.ok)
    if not audit_report.ok or audit_report.visual_findings:
        raise RuntimeError('Notebook/図表監査に未解決の指摘がある')
lifecycle.mark_completed(run_id)
final_status = lifecycle.get_run_status(run_id)
assert final_status.state == 'completed'
assert final_status.active_cell_executions == 0 and final_status.pending_notebook_writes == 0 and final_status.locks_held == 0
print('終了状態:',json.dumps(asdict(final_status),ensure_ascii=False))
lifecycle.mark_write_start(run_id)
try:
    (data_dir/'replay-phase-log.json').write_text(json.dumps(phase_log,ensure_ascii=False,indent=2),encoding='utf-8')
    (data_dir/'lifecycle-final.json').write_text(json.dumps(asdict(final_status),ensure_ascii=False,indent=2),encoding='utf-8')
finally:
    lifecycle.mark_write_end(run_id)
print('全セル終了後の保存済みNotebookは制御Notebookから再監査し、handoff-audit.jsonとNotebook metadataへ記録する。')
'''
    module_inventory = '''## 使用したJupytermindモジュール
以下は直接importし、関数・クラス・メソッドを実際に呼び出したもののみ。制御Notebookからの直接利用は明示した。内部依存関係は使用済みに数えない。

| モジュール（ai_data_scientist配下） | 直接使用したAPI | 使用目的 |
|---|---|---|
| project_manager | resolve_project, ensure_notebook, ensure_data_dir, enqueue_write | slug検証、保存先作成、データ配置、Notebookの直列書込み（セル構築は制御Notebook） |
| language_router | detect_language | 日本語判定 |
| lifecycle | register_run, mark_execution_start/end, mark_write_start/end, is_cancel_requested, mark_completed, mark_failed, get_run_status | run_idと実行・書込み・終了状態の記録 |
| ingestion | SourceSpec, ingest | 取得CSV読込み・行数上限確認 |
| cleaning | clean_dataset | 重複除去と行・列への影響確認 |
| eda | explore | 記述統計・型・非欠損件数・欠損率 |
| stats_analysis | correlation | 二値Outcomeとの点双列相関と日本語の解釈 |
| visualization | render_chart, build_image_output | 観測数棒グラフの描画とPNG MIME出力。複雑な比較図はmatplotlib＋build_image_output |
| data_definition | FieldValue, build_manifest, DataDefinitionManifest.unresolved_fields | 変数定義・単位・出典の確度。inferredは別途列挙 |
| analysis_assumptions | Assumption, AnalysisAssumptionManifest, check_manifest | 分析前提・関連スコープ・未検証前提の警告 |
| data_quality | detect_anomalies | 範囲・許容カテゴリ・欠損の意味的検査 |
| sensitivity | SensitivityPlan, run_sensitivity | 欠損処理・Winsor化・年齢形状・対象範囲の安定性確認 |
| insight_engine | extract_cited_value, record_insight | 実行出力から引用しevidence manifest付きInsightを保存（制御Notebook） |
| notebook_audit | audit_notebook(visual_audit=True) | 実行・エラー・根拠参照・図表監査（最終セルと制御Notebook） |

**非利用/適用不可**: mcp_gateway.run_and_recordはCLIツール接続をPython MCPClientとして受け渡す手段がないため未利用（CLI接続上の制約であり、本体の不具合とは断定しない）。dataset_validation.compare_datasetsおよびdata_quality.validate_anomaliesは独立参照データがないため未利用。anomaly_detectionのz-scoreは歪んだ分布を一律異常とみなすため不適用。機械学習拡張は目的外。bootstrap/GLM/スプライン/Wilson CI/VIFはnumpy/scipy/statsmodels/patsyの直接利用であり、Jupytermindの機能として数えない。
'''
    improvements = '''## Jupytermind改善候補

### 候補1：描画レイアウトによる長いラベルの切れ
**分類**: visualization.render_chartの描画品質の不具合疑い（再現済み）。**再現条件**: 初回chartsセルのbar図でPregnancies/DiabetesPedigreeFunctionなど長い列名をx目盛りにし、日本語title/xlabel/ylabelを指定。**期待する挙動**: 目盛りとxlabelが画像内に収まり読める。**実際の挙動**: fig.savefigの標準余白で下端が切れ、xlabelも見えない。**影響**: 特徴の比較と軸の意味が読めない。**回避策**: 欠損率図を横棒にし、fig.savefig(bbox_inches='tight')で余白を調整。最終成果物では解消。**関係するセル/ログ**: charts、制御Notebookのinitial-review-and-iteration-1-plan、data/initial-visual-audit.json。初回の自動visual_findingsは空であった。

### 候補2：visual_auditの目視可読性検出範囲
**分類**: notebook_auditの機能不足/拡張候補（ヒューリスティック仕様の限界で、仕様違反とは断定しない）。**再現条件**: 上記のラベル切れ画像にtitle/xlabel/ylabel/legend/missing_glyphsメタデータを付与してvisual_audit=True。**期待する挙動**: ラベル切れ・画像外の要素を警告するか、検査対象外と明示する。**実際の挙動**: chart metadataと画像密度による検査ではok=True、visual_findings=空。**影響**: 自動監査だけでは可読性を保証できない。**回避策**: 全図を目視確認し、ラベル切れと重なりを直す。**関係するセル/ログ**: 初回charts/initial-visual-audit.json。最終監査合格は人による目視確認も併用した結果。

### 他システムとの切り分け
指定ucimlのHTTP403はKaggle API/当該データセットのアクセス側の問題。対応版harsh146からSDKで取得した。Filesystem MCPの許可ディレクトリが別の場所だった問題はツール設定であり、Jupytermind本体の不具合ではない。最初に検索の先頭ページだけで指定slugがないと判断した点は分析担当側の探索不足で、修正して指定slugと対応版を照会した。ベンチマークランナーは起動していないため、その機能の評価・不具合判断は行わない。

## 初期取得履歴（最終採用データとは区別）
最初の検索確認は先頭ページだけで指定slugが見つからずLookupErrorとして扱い、Plotly CSVを2026-10-01T18:32:53.214381+00:00に一時取得した。探索を修正し、指定slugのファイルAPIも確認したがHTTPErrorとなり2026-10-01T18:33:15.873031+00:00に同CSVを再取得した。URLは https://raw.githubusercontent.com/plotly/datasets/master/diabetes.csv 、ファイルはdiabetes.csv、SHA-256は両方とも698c203a14aa31941d2251175330c9199f3ccdb31597abbba2a3e35416257a72。初期フォールバック理由は該当時点のKaggle取得失敗であり、当時の公開CSVとuciml掲載版の完全な同一性は保証されない。その後検索候補の照会を継続しharsh146版のKaggle取得に成功したため、**最終分析・全セル再実行はKaggle APIで新たに取得したファイル**を使用する。同じSHAは取得バイト列の一致のみを示し、取得できなかったuciml版との同一性を証明しない。
'''
    def prepare_final_notebook(nb):
        for identifier,source in [('acquisition',acquisition_source),('definition-cleaning',definition_source),('comparison',comparison_source)]:
            next(c for c in nb.cells if c.get('id')==identifier).source = source
        nb.cells = [c for c in nb.cells if not(c.cell_type=='markdown' and '```evidence' in c.source)]
        nb.cells += [nbformat.v4.new_markdown_cell(module_inventory),nbformat.v4.new_markdown_cell(improvements),nbformat.v4.new_code_cell(final_summary_source,id='final-summary'),nbformat.v4.new_code_cell(audit_source,id='final-audit')]
        for cell in nb.cells:
            if cell.cell_type=='code':
                cell.execution_count = None
                cell.outputs = []
        nb.metadata['reexecution'] = {'method':'restart kernel, execute every code cell in source order via Jupyter MCP', 'status':'pending'}
    write_notebook(prepare_final_notebook)
    print('最終結論・改善候補・直接利用モジュール・監査セルを保存。全コードセルの出力をクリアし再実行準備完了。')

最終結論・改善候補・直接利用モジュール・監査セルを保存。全コードセルの出力をクリアし再実行準備完了。


In [22]:
def improve_replay_logging(nb):
    setup = next(c for c in nb.cells if c.get('id')=='setup')
    setup.source = setup.source.replace("def write_notebook(mutation):\n    lifecycle.mark_write_start(run_id)", "def write_notebook(mutation):\n    phase_log.append({'phase':'notebook-write','event':'start','utc':datetime.now(timezone.utc).isoformat()})\n    lifecycle.mark_write_start(run_id)").replace("        lifecycle.mark_write_end(run_id)\nprint('言語:'", "        lifecycle.mark_write_end(run_id)\n        phase_log.append({'phase':'notebook-write','event':'end','utc':datetime.now(timezone.utc).isoformat()})\nprint('言語:'")
    for cell in nb.cells:
        if cell.cell_type=='code' and cell.id!='setup':
            cell.source = cell.source.replace("    lifecycle.mark_write_start(run_id)\n    try:", "    phase_log.append({'phase':'artifact-write','event':'start','utc':datetime.now(timezone.utc).isoformat()})\n    lifecycle.mark_write_start(run_id)\n    try:").replace("        lifecycle.mark_write_end(run_id)\n", "        lifecycle.mark_write_end(run_id)\n        phase_log.append({'phase':'artifact-write','event':'end','utc':datetime.now(timezone.utc).isoformat()})\n")
    summary_cell = next(c for c in nb.cells if c.get('id')=='final-summary')
    summary_cell.source = summary_cell.source.replace('SHA・9項目の数値照合', 'SHA・主要数値の照合')
write_notebook(improve_replay_logging)
print('書込みフェーズの開始・終了ログも再実行セルへ追加')

書込みフェーズの開始・終了ログも再実行セルへ追加


In [11]:
with phase('evidence-backed-insight-write'):
    snapshot = nbformat.read(handle.notebook_path,as_version=4)
    assert all(c.execution_count is not None for c in snapshot.cells if c.cell_type=='code' and c.id!='final-audit')
    insight_specs = [
      ('comparison',r'"Glucose_r": ([0-9.]+)', lambda v: '初回Insight・相関の解釈：Glucoseの点双列相関はr='+v+'で、この8特徴の観測値比較では最も大きい。BMI・Insulin・年齢などにも正の単変量関連があるが、列ごとに観測標本が異なり、相関は原因を示さない。','associational'),
      ('comparison',r'"Insulin_missing_pct": ([0-9.]+)', lambda v: '欠損のInsight：Insulinの欠損率は'+v+'%。ゼロ→NaNの仮定に基づく値であり、観測Insulinの関連を全標本へそのまま外挿できない。SkinThicknessも約29.6%欠損。診断基準と欠損機序は未検証。','descriptive'),
      ('adjusted-sensitivity',r'"Glucose_adjusted_OR": ([0-9.]+)', lambda v: '調整分析のInsight：Glucoseの主解析調整ORは観測SDあたり'+v+'（95%CI 2.506819–4.020279）。欠損処理/Winsor化4仕様の係数は最大相対変動0.011601、stable=True（閾値0.20）。BMIは正方向が一貫するが最大相対変動0.265176でstable=False。頑健な方向と頑健な大きさを区別する。','associational'),
      ('iteration-1',r'"complete_n": ([0-9]+)', lambda v: '反復1結果：全8特徴の完全例は'+v+'件。除外者と年齢分布が異なり、Outcome割合のFisher検定が非有意でも欠損の無作為性は証明できない。VIF最大1.616142と極端な共線性は見られず、5特徴モデルでは中央値補完/完全例のGlucose・BMIのORが近かった。残る限界は欠損選択機序と観測の独立性。','associational'),
      ('iteration-2',r'"Age_spline_p_LR": ([0-9.eE+-]+)', lambda v: '反復2結果：年齢の非線形項追加の探索的LR検定p='+v+'で適合が改善した。年齢を調整しても「妊娠回数の独立関連がある」とした単純な結論はモデル依存で、年齢スプラインではCIが1を含む。Glucose・BMIの正の関連はlog変換と年齢形状変更後も残った。残る限界は事後モデル探索・単一補完・曲線形状の因果解釈不可。','associational'),
      ('iteration-3',r'"Age60plus_n": ([0-9]+)', lambda v: '反復3結果：60歳以上は'+v+'件と少なく、高齢端の割合下降を予防効果と解釈しない。年齢形状/60歳以下を変える6仕様ではGlucoseのORが3.235987–3.339084、BMIと家族歴指標もstable=True、妊娠回数はstable=False。残る限界は高齢端の選択と一般化不能。','associational'),
      ('final-summary',r'"Pregnancies_OR_age_spline": ([0-9.]+)', lambda v: '最終Insight：年齢を非線形に扱うと妊娠回数のORは'+v+'（CIが1を含む）となり、独立した正の関連の断定を取り下げる。Glucose・BMI・家族歴指標が主な条件付き関連だが、GlucoseとOutcome定義の近接性、測定時点、欠損、未測定交絡は未解決。因果効果・将来発症・介入効果は示さない。','associational')]
    new_insight_ids = []
    for evidence_id,pattern,make_text,claim_type in insight_specs:
        evidence = next(c for c in snapshot.cells if c.id==evidence_id)
        actual_output = '\n'.join(str(value) for out in evidence.outputs for value in out.get('data',{}).values())
        cited_value = insight_engine.extract_cited_value({'output':actual_output},pattern)
        lifecycle.mark_write_start(run_id)
        try:
            insight_engine.record_insight(handle,make_text(cited_value),evidence.execution_count,cited_value,claim_type,language='ja')
        finally:
            lifecycle.mark_write_end(run_id)
        saved = nbformat.read(handle.notebook_path,as_version=4)
        new_insight_ids.append((saved.cells[-1].id,evidence_id))
    def place_insights(nb):
        mapping = dict(new_insight_ids)
        insights = {c.id:c for c in nb.cells if c.id in mapping}
        base_cells = [c for c in nb.cells if c.id not in mapping]
        ordered = []
        for cell in base_cells:
            ordered.append(cell)
            for insight_id,evidence_id in new_insight_ids:
                if evidence_id == cell.id:
                    insights[insight_id].metadata['evidence_cell_id'] = evidence_id
                    insights[insight_id].metadata['generated_by'] = 'insight_engine.record_insight'
                    ordered.append(insights[insight_id])
        nb.cells = ordered
        nb.metadata['reexecution']['status'] = 'analysis-cells-passed; final audit pending'
        nb.metadata['provenance'] = provenance
        nb.metadata['fingerprint'] = current_fingerprint
    write_notebook(place_insights)
    print('実際の再実行出力を根拠に7件のInsightを保存・隣接配置しました')
    partial_report = notebook_audit.audit_notebook(handle.notebook_path,visual_audit=True)
    print('最終監査セル実行前:',partial_report.ok,'evidence=',partial_report.insight_cell_count,'visual_findings=',partial_report.visual_findings)
    assert partial_report.ok and partial_report.insight_cell_count==7 and not partial_report.visual_findings

実際の再実行出力を根拠に7件のInsightを保存・隣接配置しました
最終監査セル実行前: True evidence= 7 visual_findings= ()


In [13]:
# 全セル終了後の実ファイルを再監査し、最終状態を永続化する。
post_report = notebook_audit.audit_notebook(handle.notebook_path,visual_audit=True)
post_status = lifecycle.get_run_status(run_id)
assert post_report.ok and not post_report.findings and not post_report.visual_findings
assert post_report.code_cell_count == post_report.executed_code_cell_count == 11
assert post_report.insight_cell_count == 7
assert post_status.state=='completed' and post_status.active_cell_executions==post_status.pending_notebook_writes==post_status.locks_held==0
saved_notebook = nbformat.read(handle.notebook_path,as_version=4)
image_count = sum('image/png' in output.get('data',{}) for cell in saved_notebook.cells if cell.cell_type=='code' for output in cell.outputs)
assert image_count == 7
executions = [{'id':cell.id,'execution_count':cell.execution_count} for cell in saved_notebook.cells if cell.cell_type=='code']
assert [item['execution_count'] for item in executions] == sorted(item['execution_count'] for item in executions)
replay_receipt = {'run_id':run_id,'status':'completed','method':'kernel restart + all 11 code cells executed sequentially with Jupyter MCP',
  'verified_at_utc':datetime.now(timezone.utc).isoformat(),'executions':executions,'image_output_count':image_count,
  'numerical_replay_match':True,'sha256_replay_match':True,'audit':asdict(post_report),'audit_ok':post_report.ok,
  'lifecycle':asdict(post_status),'manual_visual_review':'全7図を目視確認。最初のラベル切れは横棒/tight bboxで回避した。',
  'unresolved_assumption_warnings':[asdict(f) for f in assumption_findings]}
receipt_markdown = '## 再実行・引渡し監査記録\n\nカーネルを再起動し、全11コードセルを上からJupyter MCPで再実行。SHA-256と初回の主要数値は一致。7図・7件の根拠付きInsightを保存。全セル実行後のNotebook監査はok=True、未実行/エラー/visual_findingsはすべて空。ライフサイクルはcompleted、実行/書込み/ロックは0。前提manifestの3警告とデータ定義のinferred/unknownは未解決の限界として保持した。\n\n```json\n' + json.dumps(replay_receipt,ensure_ascii=False,indent=2) + '\n```\n\nセル構築とInsight保存の実コード・初期監査ログは[制御Notebook](../../../diabetes-bootstrap-v020-exp-09.ipynb)にも保存している。これは手順ログであり、再実行対象はこの分析Notebook。再実行は新しいカーネルで行い、Kaggle認証環境が必要。最後の監査セル実行途中の警告は自己監査によるもので、上記はセル終了後の確定監査結果。'
lifecycle.mark_write_start(run_id)
try:
    def persist_receipt(nb):
        nb.metadata['reexecution'] = replay_receipt
        nb.metadata['lifecycle'] = asdict(post_status)
        audit_pos = next(i for i,c in enumerate(nb.cells) if c.get('id')=='final-audit')
        nb.cells.insert(audit_pos,nbformat.v4.new_markdown_cell(receipt_markdown,id='handoff-receipt'))
    pm.enqueue_write(handle,persist_receipt)
finally:
    lifecycle.mark_write_end(run_id)
# Markdown追加後の最終実ファイルでも監査する。
verified_report = notebook_audit.audit_notebook(handle.notebook_path,visual_audit=True)
assert verified_report.ok and not verified_report.findings and not verified_report.visual_findings
assert verified_report.code_cell_count==verified_report.executed_code_cell_count==11
lifecycle.mark_write_start(run_id)
try:
    (data_dir/'handoff-audit.json').write_text(json.dumps({'reexecution':replay_receipt,'final_file_audit':asdict(verified_report),'ok':verified_report.ok},ensure_ascii=False,indent=2),encoding='utf-8')
    (data_dir/'replay-phase-log.json').write_text(json.dumps(phase_log,ensure_ascii=False,indent=2),encoding='utf-8')
finally:
    lifecycle.mark_write_end(run_id)
print('保存先:',handle.notebook_path)
print('確定監査:',verified_report.ok,'code cells=',verified_report.executed_code_cell_count,'insights=',verified_report.insight_cell_count,'PNG=',image_count)
print('確定ライフサイクル:',lifecycle.get_run_status(run_id))
print('Notebookサイズ(bytes):',handle.notebook_path.stat().st_size)

保存先: /home/nahisaho/GitHub/jupytermind/benchmark/projects/kaggle-v020-kaggle-exp-09-diabetes/notebooks/kaggle-v020-kaggle-exp-09-diabetes.ipynb
確定監査: True code cells= 11 insights= 7 PNG= 7
確定ライフサイクル: RunStatus(run_id='v020-exp-09', state='completed', active_cell_executions=0, pending_notebook_writes=0, locks_held=0, notebook_path='/home/nahisaho/GitHub/jupytermind/benchmark/projects/kaggle-v020-kaggle-exp-09-diabetes/notebooks/kaggle-v020-kaggle-exp-09-diabetes.ipynb', reason=None)
Notebookサイズ(bytes): 713058
